# Student Performance Analysis & Prediction
## Proyek Penambangan Data — EDA dan Regresi

Tujuan: mengeksplorasi dataset nilai siswa dan membandingkan model untuk memprediksi nilai akhir matematika (`G3`).

Sumber: [UCI Student Performance](https://archive.ics.uci.edu/dataset/320/student+performance). Dataset historis ini berasal dari dua sekolah di Portugal; jangan menganggap hasilnya otomatis mewakili siswa Indonesia.

In [ ]:
%pip -q install pandas numpy matplotlib seaborn scikit-learn requests

## 1. Muat dataset

In [ ]:
from pathlib import Path
from zipfile import ZipFile
from io import BytesIO
import requests
import pandas as pd

data_path = Path("student-mat.csv")
if not data_path.exists():
    url = "https://archive.ics.uci.edu/static/public/320/student+performance.zip"
    response = requests.get(url, timeout=30)
    response.raise_for_status()
    with ZipFile(BytesIO(response.content)) as archive:
        name = next(n for n in archive.namelist() if n.endswith("student-mat.csv"))
        data_path.write_bytes(archive.read(name))

df_raw = pd.read_csv(data_path, sep=";")
print("Dimensi:", df_raw.shape)
display(df_raw.head())
display(df_raw.info())

## 2. Data understanding dan EDA

In [ ]:
display(df_raw.describe(include="all").T)
display(df_raw.isna().sum().sort_values(ascending=False).to_frame("missing"))
print("Duplikasi:", df_raw.duplicated().sum())
print("Kolom:", list(df_raw.columns))

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
sns.set_theme(style="whitegrid")

fig, ax = plt.subplots(figsize=(8, 5))
sns.histplot(df_raw["G3"], bins=15, kde=True, ax=ax)
ax.set(title="Distribusi Nilai Akhir (G3)", xlabel="Nilai akhir (0–20)", ylabel="Jumlah siswa")
plt.tight_layout(); plt.show()

fig, ax = plt.subplots(figsize=(8, 5))
sns.boxplot(data=df_raw, x="studytime", y="G3", ax=ax)
ax.set(title="Nilai akhir berdasarkan kategori waktu belajar", xlabel="Kategori waktu belajar", ylabel="G3")
plt.tight_layout(); plt.show()

fig, ax = plt.subplots(figsize=(8, 5))
sns.boxplot(data=df_raw, x="failures", y="G3", ax=ax)
ax.set(title="Nilai akhir berdasarkan kode kegagalan sebelumnya", xlabel="Kode failures", ylabel="G3")
plt.tight_layout(); plt.show()

plt.figure(figsize=(11, 8))
sns.heatmap(df_raw.select_dtypes(include="number").corr(), cmap="coolwarm", center=0)
plt.title("Korelasi antarvariabel numerik"); plt.tight_layout(); plt.show()

## 3. Preprocessing

In [ ]:
df = df_raw.drop_duplicates().copy()
df["G3"] = pd.to_numeric(df["G3"], errors="coerce")
df = df.dropna(subset=["G3"]).reset_index(drop=True)
print("Baris awal:", len(df_raw))
print("Baris setelah cleaning:", len(df))
print("Missing values setelah cleaning:", int(df.isna().sum().sum()))

## 4. Modeling regresi

Kita membandingkan baseline (rata-rata), Linear Regression, Ridge, dan Random Forest. Fitur kategorikal diubah dengan OneHotEncoder; nilai numerik diimputasi dan distandardisasi melalui pipeline agar preprocessing dipelajari dari data train saja.

Dua skenario dipisahkan:
- Tanpa `G1` dan `G2`: prediksi lebih awal tanpa nilai periode sebelumnya.
- Dengan `G1` dan `G2`: prediksi ketika nilai periode sebelumnya tersedia.

Perlu diperhatikan bahwa G1 dan G2 sangat berkaitan dengan G3 menurut dokumentasi UCI.

In [ ]:
import numpy as np
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

def run_regression_experiment(data, include_previous_grades=False, random_state=42):
    target = "G3"
    excluded = [target] + ([] if include_previous_grades else ["G1", "G2"])
    X = data.drop(columns=excluded)
    y = data[target].astype(float)
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=random_state)

    numeric = X.select_dtypes(include=["number"]).columns.tolist()
    categorical = [c for c in X.columns if c not in numeric]
    preprocessor = ColumnTransformer([
        ("num", Pipeline([("imputer", SimpleImputer(strategy="median")), ("scaler", StandardScaler())]), numeric),
        ("cat", Pipeline([("imputer", SimpleImputer(strategy="most_frequent")), ("onehot", OneHotEncoder(handle_unknown="ignore"))]), categorical)
    ])
    models = {"LinearRegression": LinearRegression(), "Ridge": Ridge(alpha=1.0),
              "RandomForest": RandomForestRegressor(n_estimators=300, random_state=random_state, min_samples_leaf=2)}
    baseline = np.repeat(y_train.mean(), len(y_test))
    rows = [{"model":"MeanBaseline", "mae":mean_absolute_error(y_test, baseline),
             "rmse":mean_squared_error(y_test, baseline)**0.5, "r2":r2_score(y_test, baseline)}]
    fitted = {}
    for name, model in models.items():
        pipe = Pipeline([("preprocess", preprocessor), ("model", model)])
        pipe.fit(X_train, y_train)
        pred = pipe.predict(X_test)
        rows.append({"model":name, "mae":mean_absolute_error(y_test, pred),
                     "rmse":mean_squared_error(y_test, pred)**0.5, "r2":r2_score(y_test, pred)})
        fitted[name] = (pipe, pred, y_test)
    metrics = pd.DataFrame(rows).sort_values("mae").reset_index(drop=True)
    best = metrics[metrics["model"] != "MeanBaseline"].sort_values("mae").iloc[0]["model"]
    return metrics, fitted, best

metrics_early, fitted_early, best_early = run_regression_experiment(df, False)
metrics_late, fitted_late, best_late = run_regression_experiment(df, True)
print("Eksperimen tanpa G1/G2")
display(metrics_early)
print("Eksperimen dengan G1/G2")
display(metrics_late)
print("Model terendah MAE tanpa G1/G2:", best_early)
print("Model terendah MAE dengan G1/G2:", best_late)

## 5. Visualisasi prediksi

In [ ]:
pipe, pred, y_test = fitted_early[best_early]
plt.figure(figsize=(7, 6))
plt.scatter(y_test, pred, alpha=.75)
plt.plot([0, 20], [0, 20], linestyle="--")
plt.xlim(0, 20); plt.ylim(0, 20)
plt.xlabel("Nilai aktual G3"); plt.ylabel("Nilai prediksi G3")
plt.title(f"Aktual vs Prediksi — {best_early} (tanpa G1/G2)")
plt.tight_layout(); plt.show()

## 6. Kesimpulan

Gunakan metrik dan grafik yang dihasilkan untuk menjawab:
1. Bagaimana distribusi G3?
2. Pola apa yang tampak pada hubungan studytime/failures dengan G3?
3. Model mana yang memiliki MAE paling rendah pada test set?
4. Bagaimana perbandingan dua skenario fitur?
5. Apa batasan dataset dan mengapa hasil tidak boleh dianggap sebab-akibat?

Jangan menulis angka atau kesimpulan sebelum sel di atas benar-benar dijalankan.

## Referensi dan keterbatasan
- UCI Machine Learning Repository: [Student Performance](https://archive.ics.uci.edu/dataset/320/student+performance).
- Cortez, P. & Silva, A. (2008), *Using data mining to predict student performance*.
- Dataset historis berasal dari dua sekolah di Portugal. Performa model pada dataset ini tidak menjamin performa pada sekolah atau populasi lain.